# Exercise 3 — Anchored MC with an external generator

An **anchored MC** reproduces the conditions of a real data-taking run: filling scheme, active detectors, interaction rate, calibrations and reconstruction settings of a given pass.

The steering script `${O2DPG_ROOT}/MC/run/ANCHOR/anchorMC.sh` is configured **only through environment variables**, the same ones the GRID sets from the JDL. It:
1. finds the timestamp of the requested slice (`SPLITID`/`PRODSPLIT`) of the run;
2. derives the interaction rate and the active detectors for that time;
3. fetches the reconstruction configuration of the pass;
4. creates the workflow (`o2dpg_sim_workflow_anchored.py`) and runs it up to the AOD.

**Generator**: the official PWGHF **gap-triggered** generator. It is an *external* generator derived from Pythia8 and configured via an ini file.
Every 3rd event contains a cc̄ or bb̄ pair in |y| < 1.5 (`subGeneratorId` 0 or 1 depending on the setting). The others are minimum-bias "gap" events.

> Requires a valid AliEn token (see `00_setup.ipynb`).

In [ ]:
from swan_env import *

In [ ]:
ex3 = workdir("ex3")
os.environ["EX3"] = str(ex3)

## 3a. The generator configuration

In [ ]:
%%bash
o2exec bash -c 'cat ${O2DPG_MC_CONFIG_ROOT}/MC/config/PWGHF/ini/GeneratorHF_D2H_ccbar_and_bbbar_gap3.ini'

## 3b. The steering script

Anchoring to **run 535069** (pp 13.6 TeV, LHC23f, pass `apass2`). Kept small for SWAN: 1 timeframe, 20 signal events, 2 transport workers, QC disabled.

In [ ]:
print((PROJECT / "config" / "anchorHF.sh").read_text())

## 3c. Run it

This takes about 30–45 minutes on 4 cores, so it runs in the background. Run the `status` cell from time to time.

In [ ]:
start(f"{PROJECT}/config/anchorHF.sh", cwd=ex3, log="anchor.log")

In [ ]:
status(ex3, "anchor.log", n=25)

The first part of the log shows the anchoring decisions:

In [ ]:
%%bash
cd $EX3
grep -m20 -iE 'timestamp|interaction ?rate|detectors|colliding|SOR|EOR' anchor.log

**❓ Questions**
1. Which **interaction rate** was used? Which **detectors** are in the workflow?
2. Which **timestamp** was chosen? How does it relate to `SPLITID=50` of `PRODSPLIT=100`?
3. (After the run) change `SPLITID` (e.g. `export SPLITID=10` in a copy of the script). Do the timestamp and interaction rate change?

## 3d. Look at the AOD

When `anchor.log` reports success:

In [ ]:
%%bash
cd $EX3
AOD=$(find . -name 'AO2D*.root' | head -1); echo "AOD: $AOD"
o2exec root -l -b -q "$SIMTUT_PROJECT/macros/aodMcSummary.C(\"$AOD\",421)"

**❓** How many MC collisions come from the **charm/beauty-triggered** sub-generator, and how many are **gap** (minimum-bias) events? Is the ratio close to 1:3?

How many D⁰ per collision? Try other species: D⁺ 411, Λc⁺ 4122, B⁺ 521.

In [ ]:
%%bash
mkdir -p ~/simtutorial_results/ex3
cd $EX3 && cp $(find . -name 'AO2D*.root') anchor.log ~/simtutorial_results/ex3/ && ls ~/simtutorial_results/ex3

## 3e. (Optional) Submit the same script to the GRID

The script runs on the GRID without changes. `grid_submit.sh` sets `ALIEN_O2DPG_GRIDSUBMIT_PRODSPLIT` and `ALIEN_O2DPG_GRIDSUBMIT_SUBJOBID`, so each subjob anchors to a different slice of the run.
On the GRID, use 8 cores (`ALIEN_JDL_CPULIMIT=8`, `NWORKERS=8`) and a validated `MC-prod` release.

In [ ]:
ex3grid = workdir("ex3_grid")
start(f"$O2DPG_ROOT/GRID/utils/grid_submit.sh --script {PROJECT}/config/anchorHF.sh --jobname anchorHFtutorial "
      "--outputspec '*.log@disk=1','*.root@disk=2' "
      "--packagespec 'VO_ALICE@O2PDPSuite::MC-prod-2026-v14-1' --prodsplit 4 --wait --fetch-output",
      cwd=ex3grid, log="grid.log")

In [ ]:
status(ex3grid, "grid.log", n=20)

Follow the jobs on [MonALISA](https://alimonitor.cern.ch). When they are done, the output is downloaded into a local folder. Its name is printed in `grid.log`.

The line `**** Pipeline done success (global_runtime : …)` in the job logs gives the **CPU time per job**. This is the basis of the resource estimate needed to request an official production.

> 💡 **2-tag mode**: to generate with a new release but reconstruct with the release of the async pass, add
> `export ALIEN_JDL_O2DPG_ASYNC_RECO_TAG="VO_ALICE@O2PDPSuite::<async tag>"` to the script.

🎉 **End of the hands-on.** Questions → Mattermost `O2-simulation` / `O2DPG`, docs at <https://aliceo2group.github.io/simulation/>.